# 02 - Returns & Volatility

Understand the **statistical behaviour** of ETHUSDT daily returns.

Covers:

1. Daily returns
2. Distribution (histogram, moments)
3. Rolling volatility (annualized)
4. Drawdown
5. Rolling Sharpe
6. Monthly / yearly return tables
7. Summary stats


## Setup

In [ ]:
# --- setup --------------------------------------------------------
import sys
sys.path.insert(0, r"D:\Projects\done\eth_bot\eth_bot_v2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis.loader import load_ohlcv
from analysis import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (12, 4)

df = load_ohlcv()
close = df.close
r = close.pct_change().dropna()
print(f"days: {len(df)}   returns: {len(r)}")
print(f"range: {r.index.min().date()} -> {r.index.max().date()}")

## 1. Daily returns

Simple percent change of close-to-close.

In [ ]:
r.tail(10).to_frame("return")

## 2. Return distribution

Crypto daily returns are famously **fat-tailed** and slightly **negatively skewed**.

- kurtosis > 3 → fatter tails than normal
- skew < 0   → bigger crashes than rallies

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
r.hist(bins=80, ax=ax, color="steelblue", edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("daily returns histogram")
ax.set_xlabel("return")
plt.tight_layout()
plt.show()

In [ ]:
print(f"mean       {r.mean(): .6f}")
print(f"std        {r.std(ddof=0): .6f}")
print(f"skew       {r.skew(): .4f}")
print(f"kurtosis   {r.kurtosis(): .4f}")
print(f"min        {r.min(): .4f}")
print(f"max        {r.max(): .4f}")
print(f"P(>|5%|)   {(r.abs() > 0.05).mean():.2%}")

## 3. Rolling volatility

30-day standard deviation of returns, annualized by multiplying
by sqrt(365).

- Low vol (green zone) → calm markets, trend-following works
- High vol (red zone) → choppy, mean reversion works

In [ ]:
vol30 = r.rolling(30).std() * np.sqrt(365)

fig, ax = plt.subplots(figsize=(12, 4))
vol30.plot(ax=ax, color="darkorange", title="30d rolling volatility (annualized)")
ax.axhline(vol30.mean(), ls="--", lw=0.8, color="gray")
ax.set_ylabel("annualized vol")
plt.tight_layout()
plt.show()

print(f"mean vol 30d: {vol30.mean():.2%}")
print(f"min  vol 30d: {vol30.min():.2%}")
print(f"max  vol 30d: {vol30.max():.2%}")

## 4. Drawdown

Percent decline from the running peak.

- **Max drawdown** = worst peak-to-trough loss
- This is what an investor *feels*. Usually much larger than vol suggests.

In [ ]:
cum = (1 + r).cumprod()
peak = cum.cummax()
dd = cum / peak - 1

fig, ax = plt.subplots(figsize=(12, 4))
dd.plot(ax=ax, color="crimson", title="drawdown from peak")
ax.fill_between(dd.index, dd, 0, color="crimson", alpha=0.15)
ax.set_ylabel("drawdown")
plt.tight_layout()
plt.show()

print(f"max drawdown : {dd.min():.2%}")
print(f"when         : {dd.idxmin().date()}")

## 5. Rolling Sharpe (180d)

Sharpe = mean(return) / std(return), annualized.

Positive Sharpe = you were paid for the risk over that window.
Negative Sharpe = the window destroyed risk-adjusted value.

In [ ]:
sharpe180 = (r.rolling(180).mean() / r.rolling(180).std(ddof=0)) * np.sqrt(365)

fig, ax = plt.subplots(figsize=(12, 4))
sharpe180.plot(ax=ax, color="navy", title="rolling 180d Sharpe")
ax.axhline(0, color="black", lw=0.8)
ax.axhline(1, ls="--", color="green", lw=0.7)
ax.axhline(-1, ls="--", color="red", lw=0.7)
plt.tight_layout()
plt.show()

## 6. Monthly returns

Each cell shows the % return for that month.
Green = up month, red = down month.

In [ ]:
monthly = (1 + r).resample("M").prod() - 1
pivot = pd.DataFrame({
    "year":  monthly.index.year,
    "month": monthly.index.month,
    "ret":   monthly.values,
}).pivot(index="year", columns="month", values="ret")

pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"]

def color(v):
    if pd.isna(v): return ""
    return f"background-color: {'#d4f4d4' if v > 0 else '#f4d4d4'}"

pivot.style.format("{:+.1%}").map(color)

## 7. Yearly returns

In [ ]:
yearly = (1 + r).resample("Y").prod() - 1
yearly.index = yearly.index.year
yearly.to_frame("return").style.format("{:+.1%}")

## 8. Biggest single days

Top 5 up and top 5 down — useful for sanity checks.

In [ ]:
top_up = r.nlargest(5).to_frame("return")
top_dn = r.nsmallest(5).to_frame("return")
print("UP")
print(top_up.to_string(formatters={"return": "{:+.2%}".format}))
print("\nDOWN")
print(top_dn.to_string(formatters={"return": "{:+.2%}".format}))

## 9. Summary statistics

Everything in one dict, computed by `analysis.stats.summary()`.

In [ ]:
s = stats.summary(close)
for k, v in s.items():
    if isinstance(v, float) and abs(v) < 10:
        print(f"  {k:<14} {v: .4f}")
    else:
        print(f"  {k:<14} {v}")

## Next steps

- **Gap check** → `03_data_quality.ipynb`
- **Cross-source** → `04_funding_oi.ipynb`
- **Automated report** → `python -m analysis.reports` writes
  the same stats to `reports/daily/YYYY-MM-DD/summary.md`
